In [ ]:
import json
import re

In [82]:
with open(r"..\data\train.json") as file:
    train_data = json.load(file)

with open(r"..\data\validation.json") as file:
    val_data = json.load(file)

with open(r"..\data\test.json") as file:
    test_data = json.load(file)


len(train_data['data']), len(val_data['data']), len(test_data['data'])

(326, 82, 102)

The data across the train, val, and test dataset all need to follow the same preprocessing pieline. First we will create a function to extract important info from our dataset.

In [83]:
def extract_contracts(dataset):
    contracts = []
    for contract in dataset['data']:
        for paragraph_index, paragraph in enumerate(contract['paragraphs']):
            contracts.append({
                "contract_id": contract['title'],
                'paragraph_index': paragraph_index,
                'text': paragraph['context'],
                'qas': paragraph['qas']
            })

    return contracts

In [84]:
train_contracts = extract_contracts(train_data)
val_contracts = extract_contracts(val_data)
test_contracts = extract_contracts(test_data)

```


```

Next step is chunking contracts into smaller, more manageble size for our model (~512 token max). Each chunk should have an associated labels mapping, which indicates which clauses are inside that chunk. 

For matching labels to chunks, we will conside any overlap between an answer span and a chunk to be considered as a positive label for that class. 

The tokenizer will be kept as an input for easier replacement when exploring different models later on.

In [85]:
def create_labeled_chunks(contract, tokenizer, chunk_length=128, overlap=20):
    text = contract["text"]

    if chunk_length <= 0 or not 0 <= overlap < chunk_length:
        raise ValueError("Require chunk_length > 0 and 0 <= overlap < chunk_length")

    encoded = tokenizer(
        text,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]
    step = chunk_length - overlap

    # Keep each answer's text and its character span in the original paragraph.
    answers = []
    for qa in contract["qas"]:
        match = re.search(r'related to "([^"]+)"', qa["question"])
        if match is None:
            raise ValueError(f"Could not find category in question: {qa['question']}")

        category = match.group(1)

        for answer in qa["answers"]:
            start = answer["answer_start"]
            end = start + len(answer["text"])

            if text[start:end] != answer["text"]:
                raise ValueError(f"Answer offset does not match: {answer['text']!r}")

            answers.append((category, start, end))

    chunks = []
    for token_start in range(0, len(input_ids), step):
        token_end = min(token_start + chunk_length, len(input_ids))
        char_start = offsets[token_start][0]
        char_end = offsets[token_end - 1][1]

        labels = {
            category
            for category, answer_start, answer_end in answers
            if answer_start < char_end and char_start < answer_end # any amount of overlap is sufficient
            # if chunk_start <= answer_start and answer_end <= chunk_end # more strict, answer must be full contained in chunk
        }

        chunks.append({
            "text": text[char_start:char_end],
            "input_ids": input_ids[token_start:token_end],
            "char_start": char_start,
            "char_end": char_end,
            "labels": labels,
        })

        if token_end == len(input_ids):
            break

    return chunks

In [86]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    use_fast=True
)

In [87]:
chunks = create_labeled_chunks(train_contracts[0], tokenizer)

print(f"Chunk: {chunks[0]["text"]}")
print()
print("Tokens: ", tokenizer.decode(chunks[0]["input_ids"], skip_special_tokens=True))
print()
print(f"IDs: {chunks[0]["input_ids"]}")
print()
print(f"Labels: {chunks[0]["labels"]}")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (11248 > 512). Running this sequence through the model will result in indexing errors


Chunk: Exhibit 10.28 PRODUCT SALE AND MARKETING AGREEMENT THIS PRODUCT SALE AND MARKETING AGREEMENT (this "Agreement") is made this 12th day of November, 2018 (the "Effective Date"), by and between Calm.com, Inc., a Delaware corporation, having offices at 140 2nd Street, 3rd Floor, San Francisco, California 94105 ("Calm") and XpresSpa Group, Inc., a Delaware corporation, having offices at 780 Third Avenue, 12th Floor, New York, New York 10017 ("XSPA"). Each of Calm and XSPA may be referred to herein individually as a

Tokens:  exhibit 10. 28 product sale and marketing agreement this product sale and marketing agreement ( this " agreement " ) is made this 12th day of november, 2018 ( the " effective date " ), by and between calm. com, inc., a delaware corporation, having offices at 140 2nd street, 3rd floor, san francisco, california 94105 ( " calm " ) and xpresspa group, inc., a delaware corporation, having offices at 780 third avenue, 12th floor, new york, new york 10017 ( " xspa " ).

```


```

Next we will create the vocabulary of our labels, and assign an ID to each word. We will use this to encode our labels into a vector that the model can use.

In [88]:
label_vocab = sorted({
    re.findall(r'"([^"]+)"', qa['question'])[0]
    for contract in train_contracts
    for qa in contract['qas']
})

label_to_id = {label: i for i, label in enumerate(label_vocab)}
id_to_label = {i: label for i, label in enumerate(label_vocab)}

In [89]:
def multihot_encode(labels):
    vector = [0] * len(label_vocab)
    for label in labels:
        if label in label_to_id:
            vector[label_to_id[label]] = 1
    return vector

def multihot_decode(vector):
    labels = []
    for i, v in enumerate(vector):
        if v == 1:
            labels.append(id_to_label[i])
    return labels

In [90]:
label_vector = multihot_encode(chunks[0]['labels'])
label = multihot_decode(label_vector)

print("encoded: ", label_vector)
print("decoded: ", label)

encoded:  [0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
decoded:  ['Agreement Date', 'Document Name', 'Effective Date', 'Parties']


```


```

Final Data Preparation for Modeling

In [99]:
train_chunks = [
    chunk 
    for contract in train_contracts 
    for chunk in create_labeled_chunks(contract, tokenizer)
]

val_chunks = [
    chunk
    for contract in val_contracts
    for chunk in create_labeled_chunks(contract, tokenizer) 
]

test_chunks = [
    chunk
    for contract in test_contracts
    for chunk in create_labeled_chunks(contract, tokenizer) 
]

In [100]:
train_labels = [multihot_encode(chunk['labels']) for chunk in train_chunks]
val_labels = [multihot_encode(chunk['labels']) for chunk in val_chunks]
test_labels = [multihot_encode(chunk['labels']) for chunk in test_chunks]

In [103]:
print("Train chunk: ", len(train_chunks))
print("Train label: ", len(train_labels))
print()

print("Val chunk: ", len(val_chunks))
print("Val label: ", len(val_labels))
print()

print("Test chunk: ", len(test_chunks))
print("Test label: ", len(test_labels))

Train chunk:  35200
Train label:  35200

Val chunk:  7039
Val label:  7039

Test chunk:  9122
Test label:  9122


We will use chunk['text'] for training the TF-IDF baseline model. BERT would use the token ids found in chunk['input_ids']